### 1. Загрузка даннах

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

import warnings
warnings.simplefilter('ignore')


In [ ]:
df = pd.read_csv('../data/heart_attack_prediction_dataset.csv')

### 2. Первичное исследование: аномалии, пропуски, корректность заполнения

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna()
df.shape

In [ ]:
df.head(10)

In [ ]:
df.describe()

Базовые характеристики числовых классов нормальные

In [ ]:
# Выберу числовые не бинарные данные для визуализации распределений
df_for_visualization=df[[column for column in df.select_dtypes(include='number') if df[column].nunique()>2]]

df_for_visualization.hist(figsize=(16,9))
plt.show()

Видим что данные очень выровненные, похожие на равномерное распределение

### 3. Основное исследование данных

#### Баланс классов

In [ ]:
df['Heart Attack Risk'].value_counts()

Распределение целевого значения нормальное

Посмотрим на все столбцы с типом object и их отношение с целевой меткой

In [ ]:
for column in df:
    if df[column].dtypes == 'object':
        print(f'Колонка {column}')
        print(f'{df[column].nunique()} количество уникальных из {df[column].shape[0]}')
        print(df.groupby(['Heart Attack Risk'])[column].value_counts(), '\n')

Нет сильно выбивающихся значений

Сделаем бинарный признак для пола и разделим значение давление в два признака на нижнее - DIA, верхнее - SYS

In [ ]:
df_mod = df.copy()

In [ ]:
sex_enc = {'Male' : 0, 'Female' : 1}
df_mod['Sex'] = df['Sex'].map(sex_enc)
print(df_mod['Sex'].head(3))

In [ ]:
df_pressure = df['Blood Pressure'].str.split(pat='/', expand = True)
df_pressure.rename(columns = {0: 'Sys', 1: 'Dia'}, inplace=True)

In [ ]:
df_mod.drop(['Patient ID', 'Blood Pressure'], axis=1, inplace=True)
df_mod = pd.concat([df_mod, df_pressure], axis=1)
df_mod['Sys'] = df_mod['Sys'].astype(np.int64)
df_mod['Dia'] = df_mod['Dia'].astype(np.int64)
df_mod[['Dia', 'Sys']].head(3)

In [ ]:
def identify_categorical_features(df, unique_threshold=30, unique_ratio=0.1):

    numerical=[]
    categorical=[]
    binary = []

    ntypes = ['int64', 'float64']
    ctypes = ['object','bool']

    df_nunique = df.nunique()

    for col in df.columns:
         if (df_nunique[col] <= unique_threshold
             and df_nunique[col]/df.shape[0] <= unique_ratio):
             if df[col].dtypes in ntypes:
                 if df_nunique[col] == 2:
                     binary += [col]
                 else:
                     numerical += [col]
             elif df[col].dtypes in ctypes:
                 categorical += [col]

             elif df[col].dtypes not in category_types:
                print(f'Другой тип {col} - {df[col].dtypes}')
    return binary, numerical, categorical

In [ ]:
binary_columns, num_categorical_columns, categorical_columns = identify_categorical_features(df_mod)
num_columns = [x for x in df_mod.columns if x not in [*binary_columns, *num_categorical_columns, *categorical_columns]]

In [ ]:
print(f'Бинарные признаки{binary_columns}\nЧисловые категориальные признаки {num_categorical_columns} \nКатегорильные признаки {categorical_columns}\nЧисловые признаки {num_columns}')

Категориальных данных не так много, посмотрим на каждый из них

In [ ]:
# Diet
df_mod['Diet'].value_counts()

In [ ]:
#Всего 3 типа которые имеют логику
#можно попробовать закодировать для дальнейшего исследования Label encoder по ухудшению типа диеты
diet_map = {'Healthy':0, 'Average':1, 'Unhealthy':2}
df_mod['Diet'] = df_mod['Diet'].map(diet_map)
num_categorical_columns += ['Diet']
categorical_columns.remove('Diet')

In [ ]:
df_mod['Diet'].head(5)

In [ ]:
# Категориальные географические данные 'Country', 'Continent', 'Hemisphere'
df_mod.groupby(categorical_columns,sort=True)['Heart Attack Risk'].value_counts()

По отношению к целевой метке дисбаланса нет

#### Кореляции

Переведу label encoder географические данные, что бы посмотреть на их корреляции с остальными метками

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(dtype=int)
enc.fit(df_mod[categorical_columns])
df_mod[categorical_columns] = enc.transform(df_mod[categorical_columns])

num_categorical_columns += categorical_columns

In [ ]:
df_mod.info() #убедимся что все данные числовые

In [ ]:
df_mod.hist(figsize=(16, 16), bins=20)
plt.show()

Видно неравномерное распределение классов у признака Smoking, посмотрим на соотношение с возрастом

In [ ]:
df_mod.groupby(['Smoking'])['Age'].describe()

In [ ]:
sns.boxplot(data=df, x='Smoking', y='Age', hue='Sex')

In [ ]:
mask_male_and_smoking = (df_mod["Sex"]==0) & (df_mod["Smoking"]==1)
mask_male = (df_mod["Sex"]==0)
df_mod[mask_male_and_smoking].shape == df_mod[mask_male].shape

In [ ]:
df_mod[(df_mod['Age'] > 40) & (df_mod['Smoking'] == 0)].shape

Расределение курящих явно имеет ошибки: нет не курящих мужчин, у женщин строгое разбиение по возрасту: не курят до 40 и естьте которые курят после 40

Построим все попарные графики

In [ ]:
sns.pairplot(df_mod[num_columns[:len(num_columns)//2:] + ['Heart Attack Risk']])

In [ ]:
sns.pairplot(df_mod[num_columns[len(num_columns)//2-1::] + ['Heart Attack Risk']])
plt.show()

На попарном сравнении числовых признаков близкое к равномерному распределению

In [ ]:
corr = df_mod.corr()

mask= np.triu(np.ones_like(corr, dtype=bool))

f, ax = plt.subplots(figsize=(16, 9))


sns.heatmap(corr, mask=mask, vmax=0.5, center=0, square=True, linewidths=.5, cbar_kws={'shrink':0.5})

Из карты корреляция видна положительные корреляции курения и возраста также из-за странных данных в курении. Из-за прямой зависимости географических признаков Country, Continent, Hemisphere есть корреляции, нужно будет пробовать обучение, на наиболее содержательном признаке - стране Country

#### Мини вывод изучения данных

Распределение относительно целевой метки почти везде равномерное.
Из аномалий только странный сбор данных для категории курящих. Надо пробовать исключать эту метку для обучения.
Линейные корреляции видны для географических данных, для линейных методов надо оставить только один, попробовать разные.

### 4. Предобработка

#### Масштабирование

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
# s_scaler = StandardScaler().fit(df_mod)
# mm_scaler = MinMaxScaler().fit(df_mod)

# df_standard_scaled = s_scaler.transform(df_mod)
# df_minmax_scaled = mm_scaler.transform(df_mod)


In [ ]:
# Есть странный smoking и коррелирующие географические данные
# Выделим 4 набора
features = [*df_mod.columns.values]

no_smoking_only_country_set = [x for x in features if x!='Smoking' and x!='Continent' and x!='Hemisphere']
no_smoking_only_continent_set = [x for x in features if x!='Smoking' and x!='Country' and x!='Hemisphere']
smoking_only_country_set = [x for x in features if x!='Continent' and x!='Hemisphere']
smoking_country_and_continent_set = [x for x in features if x!='Hemisphere']
train_set = {'no_smoking_only_country_set': no_smoking_only_country_set,
               'no_smoking_only_continent_set': no_smoking_only_continent_set,
               'smoking_only_country_set': smoking_only_country_set,
               'smoking_country_and_continent_set': smoking_country_and_continent_set,
               }

### Алгоритмы, стратификация, нормализация

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#### Классификация

In [ ]:
def estimator_(df_list, df, estimator, param_grid, cat_features, byn_features=[], scoring='f1'):
    result = {}

    for el in df_list:

        y = df[df_list[el]]['Heart Attack Risk']
        X = df[df_list[el]].drop('Heart Attack Risk', axis=1)
        X_train, x_test, Y_train, y_test = train_test_split(
            X, y, random_state=322, stratify=y)

        # так как у меня есть перекодированные категориальные данные
        # разобью данные для разной отработки
        columns_for_one_hot_encoder = [column for column in X.columns if column in cat_features]
        columns_for_standardscaler = [column for column in X.columns if column not in [*cat_features, *byn_features]]

        # создадим препроцессор
        preprocessor = ColumnTransformer(
            transformers = [
                ('num', StandardScaler(), columns_for_standardscaler),
                ('cat', OneHotEncoder(drop='first'), columns_for_one_hot_encoder)
            ],
            remainder='passthrough'
        )
        
        # создадим пайплайн
        pipeline  = make_pipeline(preprocessor, estimator)

        grid_search = GridSearchCV(pipeline, param_grid, scoring=scoring)
        grid_search.fit(X_train, Y_train)

        result[el] = grid_search
    return result

In [ ]:
knn_grid = {
    'kneighborsclassifier__n_neighbors': np.array(np.linspace(1, 100, 20), dtype='int'),
    'kneighborsclassifier__metric': ['minkowski', 'euclidean'],
    }

svm_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf'],
    }

logreg_grid = {
    'logisticregression__penalty': ['l2', 'l1', 'elasticnet'],
    'logisticregression__solver': ['saga'],
    'logisticregression__C': [0.0001, 0.01, 0.1, 1, 10, 100],
    'logisticregression__l1_ratio': [0.1, 0.5, 0.9]
    }

In [ ]:
print(num_categorical_columns, '\n', categorical_columns, '\n', num_columns, '\n', binary_columns)

In [ ]:
result_for_different_sets_knn = estimator_(train_set, df_mod, 
                                           KNeighborsClassifier(), knn_grid, 
                                           num_categorical_columns, byn_features=binary_columns)

In [ ]:
result_for_different_sets_svm = estimator_(train_set, df_mod, 
                                           SVC(), svm_grid, 
                                           num_categorical_columns, byn_features=binary_columns)

In [ ]:
result_for_different_sets_logreg = estimator_(train_set, df_mod, 
                                              LogisticRegression(), logreg_grid, 
                                              num_categorical_columns, byn_features=binary_columns)

Проанализируем результаты поиска по сетке параметров для каждой модели.

In [ ]:
s=result_for_different_sets_svm['no_smoking_only_country_set'].cv_results_
tmp_fr=pd.DataFrame(s)
tmp_fr.head(12)

#### KNN

Посмотрим на значения метрик на разных наборах данных и выберем лучший из них

In [ ]:
def print_best_results_of_sets(res_of_sets):
    for el in res_of_sets:
        print (f'{el.partition('()_')[2]} '
            f'Best F1 = {res_of_sets[el].best_score_:.4f} '
            f'Parameters {res_of_sets[el].best_params_}'
           )
print_best_results_of_sets(result_for_different_sets_knn)

# for i, el in enumerate(result_for_different_sets_knn):
#     print(f'{el.removeprefix('KNeighborsClassifier()_')} Best F1 = {result_for_different_sets_knn[el].best_score_:.3f} Parameters = {result_for_different_sets_knn[el].best_params_}' )

Будем рассматривать набор где есть колонка курения и из географии только страна. Лучиший скор на метриках Минковского и при единственном соседе во всех случаях. 

In [ ]:
def plot_grid_results(grid_search, param_grid):
    name_lambda = lambda x: f'param_{x}'
    name_param = [name_lambda(x) for x in [*param_grid.keys()]]
    # преобразуем результат сетки в датафрейм
    results = pd.DataFrame(grid_search.cv_results_)
    # отберем сетку параметров и скора для двух параметров
    
    pivot_params = results.pivot(
        index = name_param[0],
        columns = name_param[1],
        values = 'mean_test_score'
    )
    fig, axes = plt.subplots(nrows=len(param_grid)+1 , ncols=1, figsize=(10, 15))
    sns.heatmap(pivot_params, annot=True, cmap='viridis', cbar_kws={'label':'F1'}, ax=axes[0])

    # построим график параметра и оценки
    for i, param in enumerate(name_param):
        sorted_results = results.sort_values(by=param)
        if sorted_results[param].nunique() > 3:
            linestyle = '--'
        else:
            linestyle = ''
        axes[i+1].errorbar(
                            x=sorted_results[param],
                            y=sorted_results['mean_test_score'],
                            yerr=sorted_results['std_test_score'],
                            capsize=5, marker='o', linestyle=linestyle
                              )
        axes[i+1].set_xlabel(param)
        axes[i+1].set_ylabel('mean_test_score')
        axes[i+1].set_title(f'f1 от {param}')
        axes[i+1].grid(True)
    pass

res1 = result_for_different_sets_knn['smoking_only_country_set']
plot_grid_results(res1, knn_grid)

**Количество соседей** - видим, что максимум достигается при 1 соседе, все следующие увеличения дали сильное падение метрики, значит данные довольно зашумленные.

**Тип подсчета рассотояния** - никак не влияет, значения абсолютно одинаковые

#### Svm

In [ ]:
print_best_results_of_sets(result_for_different_sets_svm)

Рассмотрим набор лучший набор без курения и только страной. Лучшие параметры одинаковые для всех С = 10 и rbf ядро.

In [ ]:
plot_grid_results(result_for_different_sets_svm['no_smoking_only_country_set'], svm_grid)

**Ядра**
1. Линейное ядро - из того что при любых C F1 остается нулевой означает неразделимость выборки линейной моделью и 
recol зануляется.
2. Rbf ядро - при малом штрафе - при малых C модель не находит области. При сильном штрафе начинает обводить малые области вокруг элементов.

#### LOGREG

In [106]:
print_best_results_of_sets(result_for_different_sets_logreg)

 Best F1 = 0.0059 Parameters {'logisticregression__C': 10, 'logisticregression__l1_ratio': 0.1, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}
 Best F1 = 0.0025 Parameters {'logisticregression__C': 0.1, 'logisticregression__l1_ratio': 0.1, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}
 Best F1 = 0.0059 Parameters {'logisticregression__C': 10, 'logisticregression__l1_ratio': 0.9, 'logisticregression__penalty': 'elasticnet', 'logisticregression__solver': 'saga'}
 Best F1 = 0.0059 Parameters {'logisticregression__C': 10, 'logisticregression__l1_ratio': 0.1, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}


Видим, что при любых комбинациях значение F1 крайне мало, следовательно линейные методы для этой выборки неэффективны.